## Set up docker to run the app
git pull origin main

cd docker

docker compose up --build -d

docker compose ps

# To shut down
docker compose down

## Link to Lovable app
https://assist-triage-hub.lovable.app

# Link to backend/api
http://localhost:8000/docs

# BankAssist Triage — Deployment & Setup Guide

This notebook is a **runbook for the team**. It explains how to run the finished BankAssist API using the packaged champion model.

## Default application mode

The application does **not require MLflow to be running** for normal inference.

```text
FastAPI
   ↓
models/champion_model.pkl
   ↓
TF-IDF + Logistic Regression
   ↓
intent + confidence
   ↓
routing + human-review decision
```

MLflow remains part of the separate MLOps/model-management workflow.

> Run the shell commands below in a terminal. The code cells are documentation and can be copied directly.


## 1. Repository structure required

The important deployment files should look like:

```text
bankassist-triage/
├── api/
│   ├── main.py
│   ├── model_service.py
│   ├── routing.py
│   └── schemas.py
├── models/
│   └── champion_model.pkl
├── docker/
│   ├── Dockerfile
│   └── docker-compose.yml
├── notebooks/
│   └── 04_deployment_setup.ipynb
└── requirements.txt
```

The packaged model is the default inference model so a teammate can run the application without recreating the local MLflow registry.


## 2. Get the latest project

**Run from the repository root:**


In [ ]:
git pull origin main


## 3. Start the application with Docker

**Run from the repository root:**


In [ ]:
cd docker
docker compose up --build -d


This builds and starts the FastAPI container.

Check that it is running:


In [ ]:
docker compose ps


Expected application endpoint:

- FastAPI Swagger UI: `http://localhost:8000/docs`
- Health endpoint: `http://localhost:8000/health`

A healthy response should report that the service is healthy and the model is loaded.


## 4. Test the health endpoint


In [ ]:
curl http://localhost:8000/health


Expected shape:

```json
{
  "status": "healthy",
  "model_loaded": true
}
```


## 5. Test a real prediction


In [ ]:
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"message":"I have a problem with my bank card"}'


The response should contain these four fields:

```json
{
  "intent": "...",
  "confidence": 0.0,
  "route": "...",
  "requires_human_review": true
}
```

`intent` and `confidence` come from the ML classifier. Routing and human-review decisions are returned by the backend. The frontend must not recreate this logic.


## 6. View API logs


In [ ]:
docker compose logs -f api


Press `Ctrl+C` to stop following the logs. This does **not** stop the container.


## 7. Stop the local application


In [ ]:
docker compose down


## 8. Rebuild after API, model, or dependency changes


In [ ]:
docker compose down
docker compose up --build -d


If Docker appears to be using an old cached dependency/model during troubleshooting, force a clean image rebuild from the **repository root**:


In [ ]:
docker build --no-cache -f docker/Dockerfile -t bankassist-api .


## 9. Common port conflict

If Docker reports:

```text
Bind for 0.0.0.0:8000 failed: port is already allocated
```

check what owns port 8000:


In [ ]:
sudo lsof -nP -iTCP:8000 -sTCP:LISTEN
docker ps --format "table {{.ID}}\t{{.Names}}\t{{.Ports}}" | grep 8000


Stop the old process/container before starting Compose again:

```bash
docker stop <CONTAINER_ID>
```

or, for an old local Python/Uvicorn process:

```bash
kill <PID>
```


## 10. Important model dependency

The packaged champion model was persisted with the project's pinned scikit-learn version. Keep the version in `requirements.txt` aligned with the model-training environment.

If a model loads but `predict_proba()` raises an error such as:

```text
'LogisticRegression' object has no attribute 'multi_class'
```

check the installed scikit-learn version inside the image:


In [ ]:
docker run --rm bankassist-api python -c "import sklearn; print(sklearn.__version__)"


## 11. Optional: MLflow development workflow

MLflow is used for experiment tracking, registry management and champion-model promotion. It is **not required simply to run the packaged application**.

When working on the MLflow workflow, use the team's MLflow configuration and reproduce the local registry as documented by the modelling/MLOps workflow.

The registered model URI used during development is:

```text
models:/BankAssist-Intent-Classifier@champion
```

A local API process connects to MLflow through:

```text
http://localhost:5001
```

An API container on the same Docker Compose network connects through the Docker service name:

```text
http://mlflow:5000
```

Do not commit `mlflow-data/` as a substitute for reproducing the registry.


# Production deployment — Render

The public API is hosted separately from the developer's computer.

Production architecture:

```text
Lovable frontend
      ↓ HTTPS
Render Web Service
      ↓
Docker / FastAPI
      ↓
packaged champion_model.pkl
      ↓
intent + confidence + route + human review
```

The local laptop does **not** need to remain powered on after the Render service is deployed.


## 12. Render configuration

Create a **Web Service** from the shared GitHub repository and use:

```text
Runtime: Docker
Branch: main
Dockerfile Path: docker/Dockerfile
Docker Build Context: .
Health Check Path: /health
```

Add this environment variable:

```text
MODEL_SOURCE=local
```

The Dockerfile must start Uvicorn on `0.0.0.0` and use the platform-provided `PORT` when available.

After deployment, Render provides a public HTTPS API URL. Do not use a placeholder URL in Lovable.


## 13. Verify the deployed API

Replace `<RENDER_API_URL>` with the real Render service URL:


In [ ]:
curl <RENDER_API_URL>/health

curl -X POST <RENDER_API_URL>/predict \
  -H "Content-Type: application/json" \
  -d '{"message":"I have a problem with my bank card"}'


Do not connect Lovable until both `/health` and `/predict` work using the public Render URL.


# Lovable frontend

The Lovable frontend must use the deployed Render API, not `localhost`.

Set:

```text
VITE_API_URL=<RENDER_API_URL>
```

The frontend calls:

```text
POST ${VITE_API_URL}/predict
```

Do **not** add `/predict` to `VITE_API_URL` itself.

The backend is the source of truth for:

- `intent`
- `confidence`
- `route`
- `requires_human_review`


## 14. Final verification checklist

Before the demo/submission, verify:

- Docker builds successfully.
- `docker compose up --build -d` starts the local API.
- `http://localhost:8000/health` reports `model_loaded: true`.
- Local `/predict` returns all four response fields.
- Render `/health` works publicly.
- Render `/predict` works publicly.
- Lovable uses the **real** Render URL.
- Lovable successfully classifies a query.
- No frontend routing rules or confidence thresholds duplicate backend logic.
- `mlflow-data/` is not committed.
- `requirements.txt` keeps the model-compatible scikit-learn version pinned.
